In [34]:
import matplotlib.pyplot as plt
import h5py
import numpy as np
import glob
import json
from arrakis_nd.dataset.common import *
plt.rcParams.update({'font.size': 16})
file_dir='/global/homes/n/ncarrara/scratch/MiniRun5_1E19_RHC/MiniRun5_1E19_RHC.arrakis/ARRAKIS'
# file_dir = '/global/homes/n/ncarrara/dune/segmentation_test/beta2b/'

undefined_data = []
error_data = []
num_points = []
num_events = 0

for filename in glob.glob(file_dir+'/*.ARRAKIS.hdf5'):
    try:
        file_no = filename.split(".")[-3]
        sim_h5=h5py.File(filename,'r')
        undefined = sim_h5['standard_record/undefined'][:]
        error = sim_h5['arrakis/errors'][:]
        undefined_data.append(undefined)
        num_points.append(len(sim_h5['charge/calib_prompt_hits/data']))
    except:
        pass
if len(undefined_data) > 0:
    undefined_data = np.concatenate(undefined_data)
    print(f"number of undefineds: {len(undefined_data)}")
else:
    print("no undefineds")
if len(error_data) > 0:
    error_data = np.concatenate(error_data)
    print(f"number of errors: {len(error_data)}")
else:
    print("no errors")
print("total number of points: ",sum(num_points))

number of undefineds: 50726
no errors
total number of points:  99448563


In [35]:
undefined_data.dtype

dtype([('event_id', '<i8'), ('vertex_id', '<i8'), ('traj_id', '<i8'), ('parent_id', '<i8'), ('pdg_id', '<i8'), ('parent_pdg_id', '<i8'), ('ancestor_id', '<i8'), ('ancestor_pdg_id', '<i8'), ('ancestor_level', '<i8'), ('start_process', '<i8'), ('start_subprocess', '<i8'), ('end_process', '<i8'), ('end_subprocess', '<i8'), ('parent_start_process', '<i8'), ('parent_start_subprocess', '<i8'), ('parent_end_process', '<i8'), ('parent_end_subprocess', '<i8')])

In [36]:
pdg_id = undefined_data['pdg_id']
start_process = undefined_data['start_process']
start_subprocess = undefined_data['start_subprocess']
output_data = np.vstack((pdg_id, start_process, start_subprocess)).T
unique_arrakis = np.unique(output_data, axis=0)
event_id = undefined_data['event_id']
unique_events = np.unique(event_id)
num_per_event = np.array([
    len(event_id[(event_id == event)])
    for event in unique_events
])

parent_pdg_id = undefined_data['parent_pdg_id']
ancestor_pdg_id = undefined_data['ancestor_pdg_id']
parent_start_process = undefined_data['parent_start_process']
parent_start_subprocess = undefined_data['parent_start_subprocess']

In [39]:
print(len(event_id), len(unique_events))
pdgs = []
for ii, event in enumerate(unique_events):
    if num_per_event[ii] > 1:
        pdgs.append(pdg_id[(event_id == event)])
        print(f"event: {event}, num_points: {num_per_event[ii]}")
        print(f"pdg_ids: {pdg_id[(event_id == event)]}")
print(np.unique(np.concatenate(pdgs)))

50726 46615
event: 9015, num_points: 5
pdg_ids: [3112 3112 3112 3112 3112]
event: 9025, num_points: 37
pdg_ids: [3112   11   11 3112 3112   11 3112   11 3112   11 3112   11 3112 3112
   11 3112 3112 3112   11   11 3112 3112 3112 3112 3112 3112   11 3112
 3112 3112 3112 3112 3112 3112   11   11 3112]
event: 9045, num_points: 40
pdg_ids: [11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11
 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11 11]
event: 16147, num_points: 11
pdg_ids: [11 11 11 11 11 11 11 11 11 11 11]
event: 17022, num_points: 18
pdg_ids: [  11 3112 3112 3112 3112 3112 3112 3112 3112 3112 3112 3112 3112 3112
 3112 3112 3112 3112]
event: 22054, num_points: 22
pdg_ids: [3112   11 3112 3112   11   11 3112   11 3112 3112 3112 3112 3112   11
   11 3112 3112 3112 3112 3112   11 3112]
event: 22063, num_points: 2
pdg_ids: [  11 3222]
event: 23177, num_points: 2
pdg_ids: [3222 3222]
event: 25083, num_points: 19
pdg_ids: [3112 3112 3112 3112 3112 3112 3112 3112 3112 31

In [20]:
print("unique undefineds")
print(unique_arrakis)

unique undefineds
[[     -3222          4        121]
 [      -211          1         91]
 [      -211          2          2]
 [      -211          2          3]
 [      -211          2         12]
 [      -211          2         22]
 [      -211          4        111]
 [      -211          4        121]
 [      -211          6        201]
 [       -13          1         91]
 [       -13          2          2]
 [       -13          2          3]
 [       -13          2         12]
 [       -13          2         13]
 [       -13          2         22]
 [       -13          4        111]
 [       -13          4        121]
 [       -13          6        201]
 [       -11          2          4]
 [       -11          2         14]
 [       -11          6        201]
 [        11          2          2]
 [        11          2          4]
 [        11          2         12]
 [        11          2         13]
 [        11          2         14]
 [        11          4        151]
 [        

In [21]:
for unique in unique_arrakis:
    unique_mask = (
        (pdg_id == unique[0]) & 
        (start_process == unique[1]) & 
        (start_subprocess == unique[2])
    )
    unique_parent_pdg_id = parent_pdg_id[unique_mask]
    unique_ancestor_pdg_id = ancestor_pdg_id[unique_mask]
    unique_parent_start_process = parent_start_process[unique_mask]
    unique_parent_start_subprocess = parent_start_subprocess[unique_mask]
    unique_mask_data = np.vstack((unique_parent_pdg_id, unique_parent_start_process, unique_parent_start_subprocess, unique_ancestor_pdg_id)).T
    unique_parent_data = np.unique(unique_mask_data, axis=0)
    print(f"unique instance: {unique} - number of instances: {len(unique_parent_pdg_id)}")
    for unique_ in unique_parent_data:
        unique_parent_mask = (
            (unique_parent_pdg_id == unique_[0]) & 
            (unique_parent_start_process == unique_[1]) & 
            (unique_parent_start_subprocess == unique_[2]) & 
            (unique_ancestor_pdg_id == unique_[3])
        )
        print(f"\tunique parentage: {unique_} - number of instances: {sum(unique_parent_mask)}")
    

unique instance: [-3222     4   121] - number of instances: 1
	unique parentage: [-2212     1    91 -2212] - number of instances: 1
unique instance: [-211    1   91] - number of instances: 3
	unique parentage: [  -1   -1   -1 -211] - number of instances: 3
unique instance: [-211    2    2] - number of instances: 3
	unique parentage: [  -1   -1   -1 -211] - number of instances: 3
unique instance: [-211    2    3] - number of instances: 1
	unique parentage: [  -1   -1   -1 -211] - number of instances: 1
unique instance: [-211    2   12] - number of instances: 1
	unique parentage: [  -1   -1   -1 -211] - number of instances: 1
unique instance: [-211    2   22] - number of instances: 3
	unique parentage: [  -1   -1   -1 -211] - number of instances: 3
unique instance: [-211    4  111] - number of instances: 1
	unique parentage: [  -1   -1   -1 -211] - number of instances: 1
unique instance: [-211    4  121] - number of instances: 12
	unique parentage: [-211    1   91 -211] - number of insta